# Whole Census Tract Matcher (Batches 001-007)

Resolves a single census tract per row from `4_whole_zipcode_matcher_*.csv`.

Logic:
- **One-to-One** rows: use the primary `census_tract` directly.
- **One-to-Many (Tie)** rows: extract the 6-digit tract from each entry in `tie_geoids`.
    - First, try the subset of ties that matched on ZIP. If their tracts all agree, that single tract is the resolved value.
    - If the ZIP-matching subset is empty or disagrees, fall back to checking *all* ties: if every tie shares a tract, use it.
    - Otherwise the row is flagged as `Tract Ambiguous`.

Output: split `5_whole_census_tract_matcher_*.csv` files (GitHub-size-safe), a manifest, a markdown report, and aggregate tables.

In [1]:
import os
import re
import glob
import pandas as pd
from IPython.display import Markdown, display

pd.set_option('display.max_columns', 200)

DATA_DIR = '../../data/1_derived'
INPUT_GLOB = os.path.join(DATA_DIR, '4_whole_zipcode_matcher_*.csv')
MANIFEST_INPUT = os.path.join(DATA_DIR, '4_whole_zipcode_matcher_manifest.csv')

COMBINED_OUT_PREFIX = os.path.join(DATA_DIR, '5_whole_census_tract_matcher')
COMBINED_SINGLE_OUT_PATH = os.path.join(DATA_DIR, '5_whole_census_tract_matcher.csv')
MANIFEST_OUT_PATH = os.path.join(DATA_DIR, '5_whole_census_tract_matcher_manifest.csv')
REPORT_OUT_PATH = '../../output/2_analysis/tables/5_whole_census_tract_matcher_report.md'
TABLES_OUT_DIR = '../../output/2_analysis/tables'
GITHUB_MAX_MB = 95

os.makedirs(TABLES_OUT_DIR, exist_ok=True)

all_matches = sorted(glob.glob(INPUT_GLOB))
BATCH_FILES = [
    p for p in all_matches
    if re.search(r'4_whole_zipcode_matcher_\d{3}\.csv$', os.path.basename(p))
]

if not BATCH_FILES:
    raise FileNotFoundError(f'No input batch files found for pattern: {INPUT_GLOB}')

print(f'Found {len(BATCH_FILES)} input batch files:')
for p in BATCH_FILES:
    print('  -', os.path.basename(p))

Found 7 input batch files:
  - 4_whole_zipcode_matcher_001.csv
  - 4_whole_zipcode_matcher_002.csv
  - 4_whole_zipcode_matcher_003.csv
  - 4_whole_zipcode_matcher_004.csv
  - 4_whole_zipcode_matcher_005.csv
  - 4_whole_zipcode_matcher_006.csv
  - 4_whole_zipcode_matcher_007.csv


In [2]:
parts = [pd.read_csv(p) for p in BATCH_FILES]
df = pd.concat(parts, ignore_index=True)
del parts

print(f'Loaded {len(df):,} rows from {len(BATCH_FILES)} files.')
if 'source_batch' in df.columns:
    print(df['source_batch'].value_counts().sort_index())

C:\Users\clint\AppData\Local\Temp\ipykernel_11348\3719638070.py:1: DtypeWarning: Columns (39) have mixed types. Specify dtype option on import or set low_memory=False.
  parts = [pd.read_csv(p) for p in BATCH_FILES]


C:\Users\clint\AppData\Local\Temp\ipykernel_11348\3719638070.py:1: DtypeWarning: Columns (39) have mixed types. Specify dtype option on import or set low_memory=False.
  parts = [pd.read_csv(p) for p in BATCH_FILES]


C:\Users\clint\AppData\Local\Temp\ipykernel_11348\3719638070.py:1: DtypeWarning: Columns (39) have mixed types. Specify dtype option on import or set low_memory=False.
  parts = [pd.read_csv(p) for p in BATCH_FILES]


C:\Users\clint\AppData\Local\Temp\ipykernel_11348\3719638070.py:1: DtypeWarning: Columns (39) have mixed types. Specify dtype option on import or set low_memory=False.
  parts = [pd.read_csv(p) for p in BATCH_FILES]


C:\Users\clint\AppData\Local\Temp\ipykernel_11348\3719638070.py:1: DtypeWarning: Columns (39,47,50,51) have mixed types. Specify dtype option on import or set low_memory=False.
  parts = [pd.read_csv(p) for p in BATCH_FILES]


Loaded 1,051,219 rows from 7 files.
source_batch
1     100000
2     100000
3     100000
4     100000
5     100000
6     100000
7     100000
8     100000
9     100000
10    100000
11     51219
Name: count, dtype: int64


In [3]:
def parse_pipe_list(value):
    if pd.isna(value):
        return []
    return [p.strip() for p in str(value).split('|') if p.strip()]


def normalize_tract(value):
    """Return zero-padded 6-digit census tract string, or None."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    s = str(value).strip()
    if s == '' or s.lower() == 'nan':
        return None
    if re.fullmatch(r'\d+\.0', s):
        s = s.split('.')[0]
    if not s.isdigit():
        return None
    return s.zfill(6)


def normalize_zip5(value):
    """Return 5-digit ZIP string, or None."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    s = str(value).strip()
    if s == '' or s.lower() == 'nan':
        return None
    if re.fullmatch(r'\d+\.0', s):
        s = s.split('.')[0]
    m = re.search(r'(\d{5})', s)
    return m.group(1) if m else None


def extract_tract_from_geoid(geoid):
    """Extract the 6-digit census tract from a GEOID string. Format: state(2)+county(3)+tract(6)+block(4)."""
    if geoid is None or (isinstance(geoid, float) and pd.isna(geoid)):
        return None
    s = re.sub(r'\D', '', str(geoid))
    if len(s) < 11:
        return None
    s = s.zfill(15)
    return s[5:11]


def extract_tracts_from_tie_geoids(tie_geoids):
    out = []
    for token in parse_pipe_list(tie_geoids):
        t = extract_tract_from_geoid(token)
        if t is not None:
            out.append(t)
    return out


def extract_zip_from_address(addr):
    if pd.isna(addr):
        return None
    s = str(addr)
    m_end = re.search(r'(\d{5})(?:-\d{4})?\s*$', s)
    if m_end:
        return m_end.group(1)
    m_any = re.search(r'(\d{5})', s)
    return m_any.group(1) if m_any else None

In [4]:
# Vectorized fast path for One-to-One rows; tie-specific logic only on the ~few-thousand tie rows.

df['primary_tract'] = df['census_tract'].apply(normalize_tract)

is_tie = df['match_type'] == 'One-to-Many'
n_tie = int(is_tie.sum())
n_one = len(df) - n_tie
print(f'One-to-One rows: {n_one:,}')
print(f'One-to-Many rows: {n_tie:,}')

df['tie_tracts_all_str'] = None
df['tie_tracts_zip_str'] = None
df['tie_tract_unique_count_all'] = 0
df['tie_tract_unique_count_zip'] = 0
df['tract_consistent_in_zip_matches'] = pd.NA
df['tract_consistent_in_all_ties'] = pd.NA
df['tract_resolution_method'] = None
df['final_census_tract'] = None

# One-to-One rows.
ot_o_has_tract = (~is_tie) & df['primary_tract'].notna()
ot_o_no_tract = (~is_tie) & df['primary_tract'].isna()
df.loc[ot_o_has_tract, 'tract_resolution_method'] = 'One-to-One Primary'
df.loc[ot_o_has_tract, 'final_census_tract'] = df.loc[ot_o_has_tract, 'primary_tract']
df.loc[ot_o_no_tract, 'tract_resolution_method'] = 'Tract Missing'

# One-to-Many rows: row-wise resolution on the small subset.
def resolve_tie(row):
    original_zip = normalize_zip5(row.get('zip_original'))
    tie_tracts_all = extract_tracts_from_tie_geoids(row.get('tie_geoids'))

    addresses = parse_pipe_list(row.get('tie_matched_addresses'))
    zip_idx = []
    if original_zip is not None:
        for i, addr in enumerate(addresses):
            if extract_zip_from_address(addr) == original_zip:
                zip_idx.append(i)
    tie_tracts_zip = [tie_tracts_all[i] for i in zip_idx if i < len(tie_tracts_all)]

    unique_all = sorted(set(tie_tracts_all))
    unique_zip = sorted(set(tie_tracts_zip))

    consistent_zip = (len(unique_zip) == 1) if len(tie_tracts_zip) > 0 else None
    consistent_all = (len(unique_all) == 1) if len(tie_tracts_all) > 0 else None

    if len(tie_tracts_zip) > 0 and len(unique_zip) == 1:
        method = 'Tie Resolved via ZIP-Matching Ties'
        final_tract = unique_zip[0]
    elif len(tie_tracts_all) > 0 and len(unique_all) == 1:
        method = 'Tie Resolved via All Ties Unanimous'
        final_tract = unique_all[0]
    elif len(unique_zip) > 1 or (len(unique_zip) == 0 and len(unique_all) > 1):
        method = 'Tract Ambiguous'
        final_tract = None
    else:
        method = 'Tract Missing'
        final_tract = None

    return pd.Series({
        'tie_tracts_all_str': ' | '.join(tie_tracts_all) if tie_tracts_all else None,
        'tie_tracts_zip_str': ' | '.join(tie_tracts_zip) if tie_tracts_zip else None,
        'tie_tract_unique_count_all': len(unique_all),
        'tie_tract_unique_count_zip': len(unique_zip),
        'tract_consistent_in_zip_matches': consistent_zip,
        'tract_consistent_in_all_ties': consistent_all,
        'tract_resolution_method': method,
        'final_census_tract': final_tract,
    })


if n_tie > 0:
    tie_resolved = df.loc[is_tie].apply(resolve_tie, axis=1)
    for col in tie_resolved.columns:
        df.loc[is_tie, col] = tie_resolved[col].values

df['tract_resolved'] = df['final_census_tract'].notna()
df['tract_flag'] = df['tract_resolution_method'].isin(['Tract Ambiguous', 'Tract Missing'])

print('\ntract_resolution_method counts:')
print(df['tract_resolution_method'].value_counts(dropna=False))

One-to-One rows: 1,044,522
One-to-Many rows: 6,697



tract_resolution_method counts:
tract_resolution_method
One-to-One Primary                     1043837
Tie Resolved via ZIP-Matching Ties        3746
Tract Ambiguous                           2637
Tract Missing                              685
Tie Resolved via All Ties Unanimous        314
Name: count, dtype: int64


In [5]:
tie_rows = df[is_tie].copy()
n_tie = len(tie_rows)
n_tie_zip_consistent = int((tie_rows['tract_consistent_in_zip_matches'] == True).sum())
n_tie_all_consistent = int((tie_rows['tract_consistent_in_all_ties'] == True).sum())
n_tie_resolved = int(tie_rows['tract_resolved'].sum())
n_tie_ambiguous = int((tie_rows['tract_resolution_method'] == 'Tract Ambiguous').sum())
n_tie_missing = int((tie_rows['tract_resolution_method'] == 'Tract Missing').sum())

print(f'Total One-to-Many (tie) rows:                                {n_tie:,}')
print(f'  Tie rows where all ZIP-matching ties share same tract:     {n_tie_zip_consistent:,}')
print(f'  Tie rows where ALL ties share same tract:                  {n_tie_all_consistent:,}')
print(f'  Tie rows resolved to a single census tract (final):        {n_tie_resolved:,}')
print(f'  Tie rows ambiguous (multiple distinct tracts):             {n_tie_ambiguous:,}')
print(f'  Tie rows missing tracts entirely:                          {n_tie_missing:,}')

tie_breakdown = (
    tie_rows.groupby(['zip_match_detail', 'tract_resolution_method'], dropna=False)
            .size()
            .reset_index(name='rows')
            .sort_values(['zip_match_detail', 'rows'], ascending=[True, False])
)
tie_breakdown

Total One-to-Many (tie) rows:                                6,697
  Tie rows where all ZIP-matching ties share same tract:     3,746
  Tie rows where ALL ties share same tract:                  4,045
  Tie rows resolved to a single census tract (final):        4,060
  Tie rows ambiguous (multiple distinct tracts):             2,637
  Tie rows missing tracts entirely:                          0


,zip_match_detail,tract_resolution_method,rows
0,One-to-Many Matched Exactly One Output,Tie Resolved via ZIP-Matching Ties,15
1,One-to-Many Matched Multiple Outputs,Tie Resolved via ZIP-Matching Ties,3731
2,One-to-Many Matched Multiple Outputs,Tract Ambiguous,2238
4,One-to-Many Matched No Output,Tract Ambiguous,228
3,One-to-Many Matched No Output,Tie Resolved via All Ties Unanimous,78
5,Original ZIP Missing,Tie Resolved via All Ties Unanimous,236
6,Original ZIP Missing,Tract Ambiguous,171


In [6]:
n_total = len(df)
n_resolved = int(df['tract_resolved'].sum())
n_flag = int(df['tract_flag'].sum())

method_counts = (
    df['tract_resolution_method']
    .value_counts(dropna=False)
    .rename_axis('tract_resolution_method')
    .reset_index(name='rows')
)
method_counts['pct'] = (method_counts['rows'] / n_total * 100).round(2)

print(f'Total rows:               {n_total:,}')
print(f'Resolved to single tract: {n_resolved:,} ({n_resolved/n_total*100:.2f}%)')
print(f'Flagged (ambiguous/miss): {n_flag:,} ({n_flag/n_total*100:.2f}%)')
print()
method_counts.to_csv(os.path.join(TABLES_OUT_DIR, '5_whole_census_tract_matcher_method_counts.csv'), index=False)
tie_breakdown.to_csv(os.path.join(TABLES_OUT_DIR, '5_whole_census_tract_matcher_tie_breakdown.csv'), index=False)
method_counts

Total rows:               1,051,219
Resolved to single tract: 1,047,897 (99.68%)
Flagged (ambiguous/miss): 3,322 (0.32%)



,tract_resolution_method,rows,pct
0,One-to-One Primary,1043837,99.30
1,Tie Resolved via ZIP-Matching Ties,3746,0.36
2,Tract Ambiguous,2637,0.25
3,Tract Missing,685,0.07
4,Tie Resolved via All Ties Unanimous,314,0.03


In [7]:
def estimate_rows_per_file(frame, max_bytes, sample_rows=20000, safety=0.9):
    n = len(frame)
    if n == 0:
        return 1
    sample_n = min(sample_rows, n)
    sample_csv_bytes = len(frame.head(sample_n).to_csv(index=False).encode('utf-8'))
    bytes_per_row = max(1.0, sample_csv_bytes / max(1, sample_n))
    return max(1, int((max_bytes / bytes_per_row) * safety))


def save_github_safe_csv_parts(frame, out_prefix, max_mb=95):
    max_bytes = int(max_mb * 1024 * 1024)
    n = len(frame)
    rows_per_file = estimate_rows_per_file(frame, max_bytes=max_bytes)

    part_paths = []
    manifest_rows = []
    part = 1
    start = 0

    while start < n:
        end = min(start + rows_per_file, n)
        path = f'{out_prefix}_{part:03d}.csv'
        chunk = frame.iloc[start:end]
        chunk.to_csv(path, index=False)
        size_bytes = os.path.getsize(path)

        if size_bytes > max_bytes and len(chunk) > 1:
            os.remove(path)
            rows_per_file = max(1, int((end - start) * 0.85))
            continue

        part_paths.append(path)
        manifest_rows.append({
            'part_file': os.path.basename(path),
            'rows': len(chunk),
            'size_mb': round(size_bytes / (1024 * 1024), 2),
            'start_row_1based': start + 1,
            'end_row_1based': end,
        })

        start = end
        part += 1

    manifest = pd.DataFrame(manifest_rows)
    return part_paths, manifest


def save_combined_prefer_single(frame, single_out_path, out_prefix, max_mb=95):
    max_bytes = int(max_mb * 1024 * 1024)
    frame.to_csv(single_out_path, index=False)
    single_size = os.path.getsize(single_out_path)
    if single_size <= max_bytes:
        manifest = pd.DataFrame([{
            'part_file': os.path.basename(single_out_path),
            'rows': len(frame),
            'size_mb': round(single_size / (1024 * 1024), 2),
            'start_row_1based': 1,
            'end_row_1based': len(frame),
        }])
        return [single_out_path], manifest, True
    os.remove(single_out_path)
    part_paths, manifest = save_github_safe_csv_parts(frame, out_prefix, max_mb=max_mb)
    return part_paths, manifest, False


part_paths, manifest, used_single = save_combined_prefer_single(
    df,
    single_out_path=COMBINED_SINGLE_OUT_PATH,
    out_prefix=COMBINED_OUT_PREFIX,
    max_mb=GITHUB_MAX_MB,
)
manifest.to_csv(MANIFEST_OUT_PATH, index=False)

if used_single:
    print('Saved single combined dataframe (within GitHub size limit):')
else:
    print('Single file exceeded size limit; saved GitHub-safe split files:')
for p in part_paths:
    print(f'  - {p} ({os.path.getsize(p)/(1024*1024):.2f} MB)')
print(f'Manifest: {MANIFEST_OUT_PATH}')
print(f"Total rows saved: {manifest['rows'].sum():,}")

Single file exceeded size limit; saved GitHub-safe split files:
  - ../../data/1_derived\5_whole_census_tract_matcher_001.csv (84.95 MB)
  - ../../data/1_derived\5_whole_census_tract_matcher_002.csv (84.09 MB)
  - ../../data/1_derived\5_whole_census_tract_matcher_003.csv (84.21 MB)
  - ../../data/1_derived\5_whole_census_tract_matcher_004.csv (84.71 MB)
  - ../../data/1_derived\5_whole_census_tract_matcher_005.csv (83.73 MB)
  - ../../data/1_derived\5_whole_census_tract_matcher_006.csv (83.72 MB)
  - ../../data/1_derived\5_whole_census_tract_matcher_007.csv (53.80 MB)
Manifest: ../../data/1_derived\5_whole_census_tract_matcher_manifest.csv
Total rows saved: 1,051,219


In [8]:
def dataframe_to_md_table(frame):
    if frame.empty:
        return '_No rows to display._'
    cols = [str(c) for c in frame.columns]
    header = '| ' + ' | '.join(cols) + ' |'
    sep = '| ' + ' | '.join(['---'] * len(cols)) + ' |'
    rows = []
    for _, row in frame.iterrows():
        vals = []
        for c in frame.columns:
            v = row[c]
            vals.append('' if pd.isna(v) else str(v).replace('|', '\\|'))
        rows.append('| ' + ' | '.join(vals) + ' |')
    return '\n'.join([header, sep] + rows)


report_lines = [
    '# Whole US Census Tract Match Report (Batches 001-007)',
    '',
    f"- Generated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}",
    f'- Input batch files: {len(BATCH_FILES)}',
    f'- Total rows: {n_total:,}',
    f'- Rows resolved to a single census tract: {n_resolved:,} ({n_resolved/n_total*100:.2f}%)',
    f'- Rows flagged (ambiguous or missing): {n_flag:,} ({n_flag/n_total*100:.2f}%)',
    '',
    '## Tract Resolution Method Counts',
    dataframe_to_md_table(method_counts),
    '',
    '## Tie Rows: Tract Resolution by ZIP Match Detail',
    f'Total One-to-Many (tie) rows: {n_tie:,}.',
    '',
    f'- Tie rows whose ZIP-matching ties all share the same tract: **{n_tie_zip_consistent:,}**',
    f'- Tie rows where every tie (regardless of ZIP) shares the same tract: **{n_tie_all_consistent:,}**',
    f'- Tie rows resolved to a single tract (final): **{n_tie_resolved:,}**',
    f'- Tie rows ambiguous (multiple distinct tracts): **{n_tie_ambiguous:,}**',
    f'- Tie rows missing tract data: **{n_tie_missing:,}**',
    '',
    dataframe_to_md_table(tie_breakdown),
]

report_md = '\n'.join(report_lines)
with open(REPORT_OUT_PATH, 'w', encoding='utf-8') as f:
    f.write(report_md)

display(Markdown(report_md))
print(f'\nSaved markdown report to {REPORT_OUT_PATH}')

# Whole US Census Tract Match Report (Batches 001-007)

- Generated: 2026-05-05 01:20:41
- Input batch files: 7
- Total rows: 1,051,219
- Rows resolved to a single census tract: 1,047,897 (99.68%)
- Rows flagged (ambiguous or missing): 3,322 (0.32%)

## Tract Resolution Method Counts
| tract_resolution_method | rows | pct |
| --- | --- | --- |
| One-to-One Primary | 1043837 | 99.3 |
| Tie Resolved via ZIP-Matching Ties | 3746 | 0.36 |
| Tract Ambiguous | 2637 | 0.25 |
| Tract Missing | 685 | 0.07 |
| Tie Resolved via All Ties Unanimous | 314 | 0.03 |

## Tie Rows: Tract Resolution by ZIP Match Detail
Total One-to-Many (tie) rows: 6,697.

- Tie rows whose ZIP-matching ties all share the same tract: **3,746**
- Tie rows where every tie (regardless of ZIP) shares the same tract: **4,045**
- Tie rows resolved to a single tract (final): **4,060**
- Tie rows ambiguous (multiple distinct tracts): **2,637**
- Tie rows missing tract data: **0**

| zip_match_detail | tract_resolution_method | rows |
| --- | --- | --- |
| One-to-Many Matched Exactly One Output | Tie Resolved via ZIP-Matching Ties | 15 |
| One-to-Many Matched Multiple Outputs | Tie Resolved via ZIP-Matching Ties | 3731 |
| One-to-Many Matched Multiple Outputs | Tract Ambiguous | 2238 |
| One-to-Many Matched No Output | Tract Ambiguous | 228 |
| One-to-Many Matched No Output | Tie Resolved via All Ties Unanimous | 78 |
| Original ZIP Missing | Tie Resolved via All Ties Unanimous | 236 |
| Original ZIP Missing | Tract Ambiguous | 171 |


Saved markdown report to ../../output/2_analysis/tables/5_whole_census_tract_matcher_report.md
